# Transfer Diagnostics and Unified Comparison

Feature-overlap diagnostics, shared-feature PCA projection, and unified comparison tables across experiment outputs.


In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loading import prepare_unsw_task, prepare_synthetic_task
from src.transfer import feature_overlap, pca_projection
from src.paths import ensure_dir
from src.io_utils import save_json

OUT = ensure_dir('artifacts/transfer_diagnostics')


## Feature Overlap and PCA Domain Shift


In [ ]:
task = 'multiclass'
X_unsw, y_unsw, _, _, _ = prepare_unsw_task(task)
_, _, X_syn, y_syn, _ = prepare_synthetic_task(task)

overlap = feature_overlap(X_unsw.columns, X_syn.columns)
save_json(overlap, OUT / 'feature_overlap.json')
overlap


In [ ]:
coords, domain, var_ratio, common = pca_projection(X_unsw, X_syn)
df = pd.DataFrame({'pc1': coords[:,0], 'pc2': coords[:,1], 'domain': domain})
df.to_csv(OUT / 'pca_projection.csv', index=False)

plt.figure(figsize=(6,5))
for d in sorted(df['domain'].unique()):
    sub = df[df['domain']==d]
    plt.scatter(sub['pc1'], sub['pc2'], s=6, alpha=0.5, label=d)
plt.xlabel(f'PC1 ({var_ratio[0]:.2%})')
plt.ylabel(f'PC2 ({var_ratio[1]:.2%})')
plt.title('UNSW vs Synthetic shared-feature PCA')
plt.legend()
plt.tight_layout()
plt.savefig(OUT / 'pca_projection.png', dpi=180)
plt.show()


## Available Experiment Outputs


In [ ]:
candidates = {
    'tabular_benchmarks': [
        Path('benchmark_artifacts/combined/benchmark_comparison_all.csv'),
        Path('artifacts/benchmark_comparison_all.csv'),
    ],
    'paper_cnn_unsw': [Path('cnn_benchmark_artifacts/combined/unsw_benchmark_cnn_comparison.csv')],
    'paper_cnn_transfer': [Path('cnn_benchmark_artifacts/combined/synthetic_transfer_cnn_comparison.csv')],
    'custom_or_final': [
        Path('msa_cnn_artifacts/combined/msa_cnn_drop_analysis.csv'),
        Path('final_model_artifacts/combined/final_model_results.csv'),
    ],
}

found = {}
for key, paths in candidates.items():
    for p in paths:
        if p.exists():
            found[key] = str(p)
            break
found


## Unified Comparison Table


In [ ]:
frames = []

p = Path(found.get('tabular_benchmarks', ''))
if p.exists():
    df = pd.read_csv(p)
    df['source_group'] = 'tabular_benchmarks'
    frames.append(df)

for key in ['paper_cnn_unsw', 'paper_cnn_transfer', 'custom_or_final']:
    p = Path(found.get(key, ''))
    if p.exists():
        df = pd.read_csv(p)
        df['source_group'] = key
        frames.append(df)

if frames:
    combo = pd.concat(frames, ignore_index=True, sort=False)
    combo.to_csv(OUT / 'unified_comparison.csv', index=False)
    display(combo.head(50))
else:
    print('No prior result CSVs were found yet. Run the baseline / CNN / final-model notebooks first.')


## Interpretation Inputs

Use the exported overlap, PCA, and comparison tables in the transfer discussion.
